# Chicago Affordable Housing Data Download

## Purpose

This notebook downloads and profiles Chicago's Affordable Rental Housing Developments dataset through the City of Chicago Data Portal API.

The dataset represents documented City-supported affordable developments. It does not show real-time vacancies, current rents, applicant eligibility, or every affordable unit in Chicago.

## Reproducibility

The notebook records the API endpoint and performs the download through Python so another analyst can repeat the process.

In [2]:
from pathlib import Path

import pandas as pd
import requests

In [3]:
current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

print("Current folder:", current_folder)
print("Project root:", project_root)

Current folder: /workspaces/chicago-homelessness-resource-allocation/chicago-homelessness-resource-allocation/notebooks
Project root: /workspaces/chicago-homelessness-resource-allocation/chicago-homelessness-resource-allocation


In [4]:
api_url = "https://data.cityofchicago.org/resource/s6ha-ppgi.json"
query_parameters = {"$limit": 50000}

response = requests.get(
    api_url,
    params=query_parameters,
    timeout=60,
)

response.raise_for_status()
records = response.json()
affordable_housing = pd.DataFrame(records)

print("Requested URL:", response.url)
print("HTTP status:", response.status_code)
print("Dataset dimensions:", affordable_housing.shape)

display(affordable_housing.head())

Requested URL: https://data.cityofchicago.org/resource/s6ha-ppgi.json?%24limit=50000
HTTP status: 200
Dataset dimensions: (598, 19)


,community_area,community_area_number,property_type,property_name,address,zip_code,phone_number,management_company,units,x_coordinate,y_coordinate,latitude,longitude,location,:@computed_region_awaf_s7ux,:@computed_region_43wa_7qmu,:@computed_region_vrxf_vc4k,:@computed_region_6mkv_f3dw,:@computed_region_bdys_3d7i
0,Avondale,21,Multifamily,Hairpin Lofts,3414 W. Diversey Ave.,60647,773-292-6360,Leasing & Management Co. Inc.,25,1153078.89,1918447.998,41.93207259,-87.71287204,NaN,NaN,NaN,NaN,NaN,NaN
1,Loop,32,ARO,1000M,1000 S. Michigan Ave.,60605,312-820-1000,Willow Bridge,23,1177375.505,1895971.036,41.86987759,-87.6242687,NaN,NaN,NaN,NaN,NaN,NaN
2,Logan Square,22,ARO,2556 Armtiage LLC,2556 W. Armitage Ave,60647,773-252-0600,North Clybourn Group,1,1158751.315,1913231.215,41.91764283,-87.69216996,"{'latitude': '41.917642826462', 'longitude': '...",24,41,23,22535,294
3,Douglas,35,Multifamily,South Park Plaza,2600 S. King Dr.,60616,312-674-9210,Woodlawn Comm. Dev. Corp.,134,1179206.472,1887158.196,41.84565291,-87.61781639,"{'latitude': '41.8456529117633', 'longitude': ...",1,10,1,21194,191
4,Near West Side,28,ARO,The Rosie,1461 S. Blue Island Ave.,60608,872-259-7452,The FLATS,7,1168331.384,1892984.019,41.86188118,-87.65755844,"{'latitude': '41.86188117554516', 'longitude':...",8,26,29,14920,96


In [5]:
affordable_housing.info()

profile = pd.DataFrame(
    {
        "column": affordable_housing.columns,
        "data_type": affordable_housing.dtypes.astype(str).values,
        "missing_count": affordable_housing.isna().sum().values,
        "missing_percent": (
            affordable_housing.isna().mean() * 100
        ).round(1).values,
    }
).sort_values("missing_percent", ascending=False)

print(
    "Exact duplicate rows:",
    affordable_housing.astype(str).duplicated().sum()
)

display(profile)

<class 'pandas.DataFrame'>
RangeIndex: 598 entries, 0 to 597
Data columns (total 19 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   community_area               598 non-null    str   
 1   community_area_number        598 non-null    str   
 2   property_type                598 non-null    str   
 3   property_name                598 non-null    str   
 4   address                      598 non-null    str   
 5   zip_code                     598 non-null    str   
 6   phone_number                 598 non-null    str   
 7   management_company           598 non-null    str   
 8   units                        597 non-null    str   
 9   x_coordinate                 598 non-null    str   
 10  y_coordinate                 598 non-null    str   
 11  latitude                     598 non-null    str   
 12  longitude                    598 non-null    str   
 13  location                     589 non-null    o

,column,data_type,missing_count,missing_percent
17,:@computed_region_6mkv_f3dw,str,11,1.8
16,:@computed_region_vrxf_vc4k,str,11,1.8
15,:@computed_region_43wa_7qmu,str,11,1.8
14,:@computed_region_awaf_s7ux,str,11,1.8
18,:@computed_region_bdys_3d7i,str,11,1.8
13,location,object,9,1.5
8,units,str,1,0.2
2,property_type,str,0,0.0
1,community_area_number,str,0,0.0
0,community_area,str,0,0.0


In [6]:
assert response.status_code == 200, "The API request was not successful."
assert not affordable_housing.empty, "The downloaded dataset is empty."
assert affordable_housing.columns.is_unique, "Duplicate column names were found."

raw_path = (
    project_root
    / "data"
    / "raw"
    / "affordable_rental_housing_developments.csv"
)

profile_path = (
    project_root
    / "data"
    / "processed"
    / "affordable_housing_profile.csv"
)

raw_path.parent.mkdir(parents=True, exist_ok=True)
profile_path.parent.mkdir(parents=True, exist_ok=True)

affordable_housing.to_csv(raw_path, index=False)
profile.to_csv(profile_path, index=False)

print("Download validation passed.")
print("Raw data saved to:", raw_path)
print("Profile saved to:", profile_path)

Download validation passed.
Raw data saved to: /workspaces/chicago-homelessness-resource-allocation/chicago-homelessness-resource-allocation/data/raw/affordable_rental_housing_developments.csv
Profile saved to: /workspaces/chicago-homelessness-resource-allocation/chicago-homelessness-resource-allocation/data/processed/affordable_housing_profile.csv


In [7]:
required_outputs = [raw_path, profile_path]

for path in required_outputs:
    exists = path.exists()
    size = path.stat().st_size if exists else 0

    print(path)
    print("Exists:", exists)
    print("Size in bytes:", size)
    print()

assert all(
    path.exists() and path.stat().st_size > 0
    for path in required_outputs
), "At least one required output file is missing or empty."

saved_profile = pd.read_csv(profile_path)

required_profile_columns = {
    "column",
    "data_type",
    "missing_count",
    "missing_percent",
}

assert required_profile_columns.issubset(
    saved_profile.columns
), "The saved profile is missing required columns."

print("Affordable-housing ingestion checkpoint passed.")

/workspaces/chicago-homelessness-resource-allocation/chicago-homelessness-resource-allocation/data/raw/affordable_rental_housing_developments.csv
Exists: True
Size in bytes: 198888

/workspaces/chicago-homelessness-resource-allocation/chicago-homelessness-resource-allocation/data/processed/affordable_housing_profile.csv
Exists: True
Size in bytes: 559

Affordable-housing ingestion checkpoint passed.
